# Illustrative recalibration of IE Carbon Budgets

In [ ]:
import sys
import types
import glob
import os
import re
import urllib.parse as urlp
import urllib.request as urlr
import shutil
import zipfile as zipf

#import calendar
#display(list(calendar.day_abbr))

#import pytz
import numpy as np

import matplotlib.pylab as plt
#import matplotlib.dates as mdates

import pandas as pd

In [ ]:
# display() variant that will show *all* rows and cols

# USE SPARINGLY!
# display_all() on large dataframes can cause large memory use and severe slow down on browser side!

def display_all(x) :
    with pd.option_context('display.max_rows', None,  'display.max_columns', None):
        display(x)

In [ ]:
# Adapted from: https://stackoverflow.com/questions/38358521/alternative-of-urllib-urlretrieve-in-python-3-5
def wget(url,file) :
    with urlr.urlopen(url) as response, open(file, 'wb') as out_file:
        shutil.copyfileobj(response, out_file)

## Load GWP100 factors (AR5 and AR6 variants)

+ via [Ireland's Final Greenhouse Gas Emissions 1990-2021](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-final-greenhouse-gas-emissions-1990-2021.php), table on p.30. 
+ Table manual extracted to a `.csv` file. (Note: values using "," as thousands delimited need to be enclosed in quotes, without leading space before the opening quotes.)
+ HFCs and PFCs are not broken out individually, but shown with a range (for both AR4 and AR5). For other gases there is just a single value (shown in the "low" columns).
+ For the current "illustrative example" of recalibrating CBs I am going to ignore AR4-5 adjustments for HFCs and PFCs: so strip out those gases (and now redundant "low-high" column labels).

In [ ]:
gwp100 = pd.read_csv("GWPs-p30.csv", header=0, index_col=1, thousands=',')
gwp100 = gwp100.drop(['CO2','HFCs','PFCs'], axis='index') 
  # CO2 GWP100 = 1.0 by definition; HFCs and PFCs not unbundled, so also not usable for current purposes 
gwp100 = gwp100.drop(['Industrial designation or common name', 'AR4-high', 'AR5-high'],axis='columns')
gwp100 = gwp100.rename({'AR4-low':'AR4','AR5-low':'AR5'},axis='columns')
gwp100['AR4-5-rescale']  = gwp100['AR5']/gwp100['AR4']
display(gwp100)

## Locating NIR/CRF data

We *have* to use the NIR data (rather than domestic "GHG Emissions" report) at least up to the 2022 cycle (1990-2020), because up to and including this point, EPA were not including LULUCF in the domestic report. It is a bit more tedious than the domestic report because the NIR CRF uses a separate file for each year, rather than the integrated single-file format of the domestic report. There again, it includes more detail, and is arguably the "authoritative" data.


# UNFCCC NIR page links (all parties) and Ireland IE CRF zipfile links

Manually harvested. 

+ [2026](https://unfccc.int/ghg-inventories-annex-i-parties/2026)
    + [IE CRF zip](https://unfccc.int/sites/default/files/resource/IRL-CRT-2026-V1.0.zip)
+ [2025](https://unfccc.int/ghg-inventories-annex-i-parties/2025)
    + [IE CRF zip](https://unfccc.int/sites/default/files/resource/IRL-CRT-2025-V1.2-20250314-111705_awaiting%20approval.zip)
+ [2024](https://unfccc.int/ghg-inventories-annex-i-parties/2024)
    + [IE CRF zip](https://unfccc.int/sites/default/files/resource/IRL-CRT-2024-V1.0-1990-2022-20241217-102401.zip) 
+ [2023](https://unfccc.int/ghg-inventories-annex-i-parties/2023)
    + [IE CRF zip](https://unfccc.int/sites/default/files/resource/irl-2023-crf-14apr23_AR5.zip) 
+ [2022](https://unfccc.int/ghg-inventories-annex-i-parties/2022)
    + [IE CRF zip](https://unfccc.int/sites/default/files/resource/irl-2022-crf-7sep22.zip)
+ [2021](https://unfccc.int/ghg-inventories-annex-i-parties/2021)
    + [IE CRF zip](https://unfccc.int/sites/default/files/resource/irl-2021-crf-13apr21.zip)
+ [2020](https://unfccc.int/ghg-inventories-annex-i-parties/2020)
    + [IE CRF zip](https://unfccc.int/sites/default/files/resource/irl-2020-crf-15apr20.zip)
+ [2019](https://unfccc.int/process-and-meetings/transparency-and-reporting/reporting-and-review/reporting-and-review-under-the-convention/greenhouse-gas-inventories-annex-i-parties/submissions/national-inventory-submissions-2019)
    + [IE CRF zip](https://unfccc.int/sites/default/files/resource/irl-2019-crf-10apr19.zip)
+ [2018](https://unfccc.int/process-and-meetings/transparency-and-reporting/reporting-and-review/reporting-and-review-under-the-convention/greenhouse-gas-inventories-annex-i-parties/submissions/national-inventory-submissions-2018)
    + [IE CRF zip](https://unfccc.int/sites/default/files/resource/irl-2018-crf-6nov18.zip)
+ [2017](https://unfccc.int/process-and-meetings/transparency-and-reporting/reporting-and-review/reporting-and-review-under-the-convention/greenhouse-gas-inventories-annex-i-parties/submissions/national-inventory-submissions-2017)
    + [IE CRF zip](https://unfccc.int/files/national_reports/annex_i_ghg_inventories/national_inventories_submissions/application/zip/irl-2017-crf-12apr17.zip)


In [ ]:
# Manually harvested
unfccc_crf_urls = [
    'https://unfccc.int/sites/default/files/resource/IRL-CRT-2026-V1.0.zip',
    'https://unfccc.int/sites/default/files/resource/IRL-CRT-2025-V1.2-20250314-111705_awaiting%20approval.zip',
    'https://unfccc.int/sites/default/files/resource/IRL-CRT-2024-V1.0-1990-2022-20241217-102401.zip',
    'https://unfccc.int/sites/default/files/resource/irl-2023-crf-14apr23_AR5.zip',
    'https://unfccc.int/sites/default/files/resource/irl-2022-crf-7sep22.zip',
    'https://unfccc.int/sites/default/files/resource/irl-2021-crf-13apr21.zip',
    'https://unfccc.int/sites/default/files/resource/irl-2020-crf-15apr20.zip',
    'https://unfccc.int/sites/default/files/resource/irl-2019-crf-10apr19.zip',
    'https://unfccc.int/sites/default/files/resource/irl-2018-crf-6nov18.zip',
    'https://unfccc.int/files/national_reports/annex_i_ghg_inventories/national_inventories_submissions/application/zip/irl-2017-crf-12apr17.zip']

unfccc_crf_zipfiles = pd.DataFrame(columns = ['url','zipfile','GWP100s'])
for u in unfccc_crf_urls :
    url_split = urlp.urlsplit(u)
    url_path_split = url_split.path.split('/')
    zipfile = url_path_split[-1]
    year = int(re.search(r'[^0-9]*([0-9]{4})', zipfile).group(1))
    if (year >= 2023):
        gwp100s = 'AR5'
    else :
        gwp100s = 'AR4'
    unfccc_crf_zipfiles.loc[year] = [u,zipfile,gwp100s]

In [ ]:
display(crf_zipfiles)

#### But ... UNFCCC CRF links not usable in python?

Sadly, while the CRF links work fine within a (firefox) browser, they don't work via python - not clear what the issue is, but presumably something to do with blocking robot scraping etc. Urg.

There are some indications of attempts at an API for UNFCCC GHG data, but nothing that seems currently stable/working.

+ [UNFCCC: GHG Data Interface](https://ghgdi.unfccc.int/)
+ [zenodo: UNFCCC country-submitted greenhouse gas emissions data until 2024-07-04](https://zenodo.org/records/12657469)
+ [unfccc-di-api 4.0.4](https://pypi.org/project/unfccc-di-api/)


### epa.ie NIR landing page and CRF zipfile links [Ireland only]

Manually harvested. Can't find anything earlier than 2019 - doh!

+ [2026](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-national-inventory-submissions-2026.php)
    + [IE CRF zip](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL-CRT-2026-V1.0.zip)
+ [2025](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-national-inventory-submissions-2025.php)
    + [IE CRF zip](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/Ireland-Common-Reporting-Tables-(CRT)-2025.zip)
+ [2024](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-national-inventory-submissions-2024.php)
    + [IE CRF zip](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL_2024__12032024_130024_started.zip)
+ [2023](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-national-inventory-submissions-2023.php)
    + [IE CRF zip](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL_2023_14042023_160711.zip) 
+ [2022](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-national-inventory-submissions-2022.php)
    + [IE CRF zip](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL_2022_06092022_174936_website.zip)
+ [2021](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-national-inventory-submissions-2021.php)
    + [IE CRF zip](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL_2021__10032021_160609_started.zip)
+ [2020](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-national-inventory-submissions-2020.php)
    + [IE CRF zip](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL_2020_2015_06032020_151848_started-(1).zip)
+ [2019](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-national-inventory-submissions-2019.php)
    + [IE CRF zip](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/Submission-files_CRF.zip)


In [ ]:
# Manually harvested
epa_crf_urls = [
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL-CRT-2026-V1.0.zip',
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/Ireland-Common-Reporting-Tables-(CRT)-2025.zip',
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL_2024__12032024_130024_started.zip',
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL_2023_14042023_160711.zip',
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL_2022_06092022_174936_website.zip',
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL_2021__10032021_160609_started.zip',
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL_2020_2015_06032020_151848_started-(1).zip',
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/Submission-files_CRF.zip']

crf_root = 'indata/crf'
epa_crf_zipfiles = pd.DataFrame(columns = ['url','folder','zipfile','zippath','GWP100s','skiprows','index_col'])
for u in epa_crf_urls :
    url_split = urlp.urlsplit(u)
    url_path_split = url_split.path.split('/')
    zipfile = url_path_split[-1]
    try :
        year = int(re.search(r'[^0-9]*([0-9]{4})', zipfile).group(1))
    except : 
        year = 2019 # VERY UGLY AND FRAGILE HACK!
    if (year >= 2023):
        gwp100s = 'AR5'
    else :
        gwp100s = 'AR4'
    if (year >= 2025) :
        skiprows = 7
        index_col = 1
    else :
        skiprows = 4
        index_col = 0
    folder = crf_root+'/'+str(year)
    zippath = folder+'/'+zipfile
    epa_crf_zipfiles.loc[year] = [u,folder,zipfile,zippath,gwp100s,skiprows,index_col]

In [ ]:
display(epa_crf_zipfiles)

## Download and unzip NIR/CRF files

In [ ]:
def get_unzip_crf(year) :
    src = epa_crf_zipfiles.loc[year,:]

    if (os.path.exists(src["folder"])) :
        print(F'crf folder already exists: {src["folder"]}')
    else :
        os.mkdir(src['folder'])

    if(os.path.exists(src['zippath'])) :
        print(F'crf zip file already exists: {src["zippath"]}')
    else :
        wget(src['url'],src['zippath'])

    with zipf.ZipFile(src['zippath'], 'r') as zip_ref:
        zip_ref.extractall(src['folder'])
    

In [ ]:
for year in epa_crf_zipfiles.index :
    print(year)
    get_unzip_crf(year)

## Load and preprocess NIR-CRF data

Up to and including the 2022 cycle (covering 1990-2020), the EPA were still using AR4 GWP100 values for aggregation. Given that the CCAC already "manually" updated for AR5 values even on the first cycle of the CB programme, we need to already make that adjustment here, where applicable.

In [ ]:
def load_single_invfile(invfile, skiprows, index_col, gwp100s) :

    raw_totals = pd.read_excel(invfile, 
        sheet_name="Summary2", 
        skiprows=skiprows,
        index_col=index_col,
        nrows=2, header=0).transpose()

    # Drop empty leading columns (if any); transposed to rows here...
    raw_totals = raw_totals.iloc[index_col :, :]

    # Flatten minor(?) variations in row and column names across CRF years
    #print(raw_totals[year].columns)
    for colname in raw_totals.columns :
        if re.match(".*Total.*net emissions.*",colname) :
            raw_totals = raw_totals.rename(columns={colname:'Total (net emissions)'})

    for rowname in raw_totals.index :
        if re.match("CO2",rowname) :
            raw_totals = raw_totals.rename(index={rowname:'CO2'})

    raw_totals = raw_totals.drop('Unspecified mix of HFCs and PFCs',axis='index') # All "NO"
    
    return(raw_totals)

In [ ]:
def parse_crf_fileyear(crf_file) :
    path_split = crf_file.split('/')
    filename = path_split[-1]

    try :
      inventory_year = int(re.search('^.*[_-][\d]{4}.*[_-]([\d]{4})[^\d].*xlsx$', filename).group(1))
    except Exception as err:
        print(f"Unexpected {err=}, {type(err)=}")
        inventory_year = 0
    #print(F'filename: {filename} inventory_year: {inventory_year}')
    return (inventory_year)


In [ ]:
def load_single_crf(crf_zipfiles, crfyear) :
    crf_invyear_filepattern = crf_root+'/'+str(crfyear)+"/IRL*.xlsx"

    crf_invyear_files = pd.DataFrame(glob.glob(crf_invyear_filepattern), columns=['file'])
        # glob to avoid picking up extraneous files, such as lock files if some files open in calc application!
    
    crf_invyears = crf_invyear_files['file'].apply(parse_crf_fileyear)
    
    crf_invyears.name = 'invyear'
    crf_invyear_files.index = crf_invyears
    crf_invyear_files = crf_invyear_files.sort_index()
    crf_invyear_files = crf_invyear_files.loc[2015:,:] # Retrict to CBref years and beyond

    raw_totals = {}
    for invyear in crf_invyear_files.index :
        #print(F'invyear: {invyear}')
        raw_totals[invyear] = load_single_invfile(
            invfile=crf_invyear_files.at[invyear,'file'],
            skiprows=crf_zipfiles.loc[crfyear,'skiprows'], 
            index_col=crf_zipfiles.loc[crfyear,'index_col'], 
            gwp100s=crf_zipfiles.loc[crfyear,'GWP100s'])
    
    crf_df = pd.DataFrame()
    invyear0 = list(raw_totals.keys())[0] # Assume indices are the same for all invyears in any single crfyear!
    crf_df.index = raw_totals[invyear0].index

    for invyear in crf_invyear_files.index :
        crf_df[invyear] = raw_totals[invyear]['Total (net emissions)']
    
    crf_df.loc['Total-xchk'] = crf_df['CO2':'NF3'].sum(axis='index')
    
    return crf_df

In [ ]:
def load_crf_data(crf_zipfiles) :

    crf_data = {} 
    for crf_year in crf_zipfiles.index :
        print(F'crf_year: {crf_year}')
        crf_data[crf_year] = load_single_crf(crf_zipfiles,crf_year)

    return(crf_data)

In [ ]:
crf_data_raw = load_crf_data(epa_crf_zipfiles)

In [ ]:
for crf_year in crf_data_raw.keys() :
    print(F'crf_year: {crf_year}')
    display(crf_data_raw[crf_year])

In [ ]:
def rescale_ar4_ar5(inv_data) :
    for gas in gwp100.index :
        rescale = gwp100.loc[gas, 'AR4-5-rescale']
        inv_data.loc[gas,:] *= rescale
    inv_data.loc['Total-xchk',:] = inv_data['CO2':'NF3'].sum(axis='index')
    return(inv_data)

In [ ]:
crf_data_ar5 = {}
for crf_year in crf_data_raw.keys() :
    crf_data_ar5[crf_year] = crf_data_raw[crf_year].copy()
    
    if (epa_crf_zipfiles.at[crf_year,'GWP100s'] == 'AR4') :
        crf_data_ar5[crf_year] = rescale_ar4_ar5(crf_data_ar5[crf_year])


In [ ]:
for crf_year in crf_data_ar5.keys() :
    print(F"crf_year: {crf_year} GWP100s: {epa_crf_zipfiles.at[crf_year,'GWP100s']}")
    display(crf_data_ar5[crf_year])

## Extract/summarise CRF data for "statutory" reference inventory year (2018)

In [ ]:
reference_year = 2018
reference_data = pd.DataFrame()
for crf_year in crf_data_ar5.keys() :
    if (crf_year >= (reference_year+2)) :
        reference_data[crf_year] = crf_data_ar5[crf_year].loc['CO2':'NF3',reference_year]/1000.0 # inc. rescale kt -> Mt

reference_data = reference_data.sort_index(axis='columns',ascending=True)

In [ ]:
reference_data.loc['Total CO2e (AR5)',:] = reference_data.sum(axis='index')

reference_data_benchmark = reference_data.at['Total CO2e (AR5)',2021]
reference_data.loc['Rescale',:] = reference_data.loc['Total CO2e (AR5)',:]/reference_data_benchmark

In [ ]:
display(reference_data)

In [ ]:
cb_cycle1 = pd.DataFrame(columns=reference_data.columns)
cb_cycle1.loc['reference_data',:] = reference_data.loc['Total CO2e (AR5)',:]

# CB cycle 1 as originally proposed from:
# https://www.climatecouncil.ie/media/climatechangeadvisorycouncil/Technical%20report%20on%20carbon%20budgets%2025.10.2021.pdf
cb_cycle1.loc['CB1',:] = 295.0 * reference_data.loc['Rescale']
cb_cycle1.loc['CB2',:] = 200.0 * reference_data.loc['Rescale']
cb_cycle1.loc['CB3',:] = 151.0 * reference_data.loc['Rescale']

In [ ]:
display(cb_cycle1)

## Extract/summarise CRF data for "notional" CB cycle 1 reference period (2015-2019)

Roughly speaking (!) this is the latest 5 year period for which a CRF was available prior to the CCAC finalising their cycle 1 proposals.

In [ ]:
cb_ref = pd.DataFrame()
for crf_year in crf_data_ar5.keys() :
    if (crf_year >= 2021) :
        cb_ref[crf_year] = crf_data_ar5[crf_year].loc['CO2':'NF3',2015:2019].sum(axis='columns')/1000.0 # inc. rescale kt -> Mt

cb_ref = cb_ref.sort_index(axis='columns',ascending=True)

In [ ]:
cb_ref.loc['Total CO2e (AR5)',:] = cb_ref.sum(axis='index')

cb_ref_benchmark = cb_ref.at['Total CO2e (AR5)',2021]
cb_ref.loc['Rescale',:] = cb_ref.loc['Total CO2e (AR5)',:]/cb_ref_benchmark

In [ ]:
display(cb_ref)

In [ ]:
cb_cycle1 = pd.DataFrame(columns=cb_ref.columns)
cb_cycle1.loc['cb_ref',:] = cb_ref.loc['Total CO2e (AR5)',:]

# CB cycle 1 as originally proposed from:
# https://www.climatecouncil.ie/media/climatechangeadvisorycouncil/Technical%20report%20on%20carbon%20budgets%2025.10.2021.pdf
cb_cycle1.loc['CB1',:] = 295.0 * cb_ref.loc['Rescale']
cb_cycle1.loc['CB2',:] = 200.0 * cb_ref.loc['Rescale']
cb_cycle1.loc['CB3',:] = 151.0 * cb_ref.loc['Rescale']

In [ ]:
display(cb_cycle1)

## Extract/summarise CRF data for CB0 period

In [ ]:
cb0 = pd.DataFrame()
for crf_year in crf_data_ar5.keys() :
    if (crf_year >= 2022) :
        cb0[crf_year] = crf_data_ar5[crf_year].loc['CO2':'NF3',2016:2020].sum(axis='columns')/1000.0 # inc. rescale kt -> Mt

cb0 = cb0.sort_index(axis='columns',ascending=True)

In [ ]:
cb0.loc['Total CO2e (AR5)',:] = cb0.sum(axis='index')

cb0_benchmark = cb0.at['Total CO2e (AR5)',2022]
cb0.loc['Rescale',:] = cb0.loc['Total CO2e (AR5)',:]/cb0_benchmark

In [ ]:
display(cb0)

In [ ]:
cb_cycle1 = pd.DataFrame(columns=cb0.columns)
cb_cycle1.loc['CB0',:] = cb0.loc['Total CO2e (AR5)',:]

# CB cycle 1 as originally proposed from:
# https://www.climatecouncil.ie/media/climatechangeadvisorycouncil/Technical%20report%20on%20carbon%20budgets%2025.10.2021.pdf
cb_cycle1.loc['CB1',:] = 295.0 * cb0.loc['Rescale']
cb_cycle1.loc['CB2',:] = 200.0 * cb0.loc['Rescale']
cb_cycle1.loc['CB3',:] = 151.0 * cb0.loc['Rescale']

In [ ]:
display(cb_cycle1)

## Load and preprocess EPA Provisional Inventory data

WORK IN PROGRESS!

### epa.ie Provisional Inventory landing page and datafile links [Ireland only]

Manually harvested.

+ [Ireland's Provisional Greenhouse Gas Emissions 1990-2025](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-provisional-greenhouse-gas-emissions-1990-2025.php)
    + [IE pinv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional_Emissions_1990-2025_web.xlsx) 
+ [Ireland's Provisional Greenhouse Gas Emissions 1990-2024](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-provisional-greenhouse-gas-emissions-1990-2024.php)
    + [IE pinv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional_Emissions_1990-2024_web.xlsx) 
+ [Ireland's Provisional Greenhouse Gas Emissions 1990-2024](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-provisional-greenhouse-gas-emissions-1990-2023.php)
    + [IE pinv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional_Emissions_1990-2023_web.xlsx) 
+ [Ireland's Provisional Greenhouse Gas Emissions 1990-2024](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-provisional-greenhouse-gas-emissions-1990-2022.php)
    + [IE pinv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional-emissions-1990-2022_Web.xlsx) 
+ [Ireland's Provisional Greenhouse Gas Emissions 1990-2024](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-provisional-greenhouse-gas-emissions-1990-2021.php)
    + [IE pinv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Final-emissions-data_1990-2021_AR5_Web.xlsx) (File name states "final" and "AR5", suggesting that it might actually be the 2023 "final" emissions report; but file properties show date of "21/07/2022, 11:49:44" so it *is* the 2022 "provisional" report, and the data does actually differ from the actual 2023 "final" report. Doh!)
+ [Ireland's Provisional Greenhouse Gas Emissions 1990-2024](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-provisional-greenhouse-gas-emissions-1990-2020.php)
    + [IE pinv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional-emissions_1990-2020_web.xlsx) 
+ [Ireland's Provisional Greenhouse Gas Emissions 1990-2024](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-provisional-greenhouse-gas-emissions-1990-2019.php)
    + [IE pinv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional_emissions_1990-2019_web.xlsx) 


In [ ]:
# Manually harvested
epa_pinv_urls = [
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional_Emissions_1990-2025_web.xlsx', 
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional_Emissions_1990-2024_web.xlsx', 
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional_Emissions_1990-2023_web.xlsx', 
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional-emissions-1990-2022_Web.xlsx', 
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Final-emissions-data_1990-2021_AR5_Web.xlsx', 
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional-emissions_1990-2020_web.xlsx', 
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional_emissions_1990-2019_web.xlsx']

pinv_root = 'indata/pinv'
epa_pinv_xlsxfiles = pd.DataFrame(columns = ['url','folder','xlsxfile','xlsxpath','GWP100s','skiprows','index_col'])
for u in epa_pinv_urls :
    url_split = urlp.urlsplit(u)
    url_path_split = url_split.path.split('/')
    xlsxfile = url_path_split[-1]
    try :
        pinv_year = int(re.search(r'[^\d]*[\d]{4}[-_]([0-9]{4})', xlsxfile).group(1)) + 1
    except Exception as err:
        print(f"Unexpected {err=}, {type(err)=}")
        pinv_year = 0
    if (pinv_year >= 2022):
        gwp100s = 'AR5'
    else :
        gwp100s = 'AR4'

    # WIP: to be checked/corrected
    if (pinv_year >= 2025) :
        skiprows = 7
        index_col = 1
    else :
        skiprows = 4
        index_col = 0
    folder = pinv_root+'/'+str(pinv_year)
    xlsxpath = folder+'/'+xlsxfile
    epa_pinv_xlsxfiles.loc[pinv_year] = [u,folder,xlsxfile,xlsxpath,gwp100s,skiprows,index_col]

In [ ]:
display(epa_pinv_xlsxfiles)

## Load and preprocess EPA Final Inventory data

WORK IN PROGRESS!

### epa.ie Final Inventory landing page and datafile links [Ireland only]

Manually harvested.

+ [Ireland's Final Greenhouse Gas Emissions 1990-2025](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-final-greenhouse-gas-emissions-1990-2024.php)
    + [IE finv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG-Final-emissions-data-1990-2024_Web.xlsx) 
+ [Ireland's Final Greenhouse Gas Emissions 1990-2025](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-final-greenhouse-gas-emissions-1990-2023.php)
    + [IE finv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG-Final-emissions-data-1990-2023_Web.xlsx) 
+ [Ireland's Final Greenhouse Gas Emissions 1990-2025](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-final-greenhouse-gas-emissions-1990-2022.php)
    + [IE finv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Final-emissions-data_1990-2022_Web.xlsx) 
+ [Ireland's Final Greenhouse Gas Emissions 1990-2025](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-final-greenhouse-gas-emissions-1990-2021.php)
    + [IE finv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Final-emissions-data_1990-2021_Webv2.xlsx) 



In [ ]:
# Manually harvested
epa_finv_urls = [
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG-Final-emissions-data-1990-2024_Web.xlsx', 
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG-Final-emissions-data-1990-2023_Web.xlsx', 
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Final-emissions-data_1990-2022_Web.xlsx', 
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Final-emissions-data_1990-2021_Webv2.xlsx']

finv_root = 'indata/finv'
epa_finv_xlsxfiles = pd.DataFrame(columns = ['url','folder','xlsxfile','xlsxpath','GWP100s','skiprows','index_col'])
for u in epa_finv_urls :
    url_split = urlp.urlsplit(u)
    url_path_split = url_split.path.split('/')
    xlsxfile = url_path_split[-1]
    try :
        finv_year = int(re.search(r'[^\d]*[\d]{4}[-_]([0-9]{4})', xlsxfile).group(1)) + 2
    except Exception as err:
        print(f"Unexpected {err=}, {type(err)=}")
        finv_year = 0
    if (finv_year >= 2023):
        gwp100s = 'AR5'
    else :
        gwp100s = 'AR4'

    # WIP: to be checked/corrected
    if (finv_year >= 2025) :
        skiprows = 7
        index_col = 1
    else :
        skiprows = 4
        index_col = 0
    folder = finv_root+'/'+str(finv_year)
    xlsxpath = folder+'/'+xlsxfile
    epa_finv_xlsxfiles.loc[finv_year] = [u,folder,xlsxfile,xlsxpath,gwp100s,skiprows,index_col]

In [ ]:
display(epa_finv_xlsxfiles)

In [ ]:
crf_2022_cb0 = pd.DataFrame()
crf_2022_cb0.index = raw_totals[2016].index

for year in crf_2022_cb0_files.index :
    crf_2022_cb0[year] = raw_totals[year]['Total (net emissions)(1)']

crf_2022_cb0 = crf_2022_cb0.drop('Unspecified mix of HFCs and PFCs',axis='index') # All "NO"
crf_2022_cb0 = crf_2022_cb0.rename({'CO2(1)':'CO2'},axis='index')

crf_2022_cb0.loc['Total-xchk'] = crf_2022_cb0.drop('Total ',axis='index').sum(axis='index')

crf_2022_cb0['CB0'] = crf_2022_cb0.sum(axis='columns')

In [ ]:
display(crf_2022_cb0)

In [ ]:
# Checkpoint
crf_2022_cb0_chkpt = crf_2022_cb0.copy()

In [ ]:
# Restore
crf_2022_cb0 = crf_2022_cb0_chkpt.copy()

In [ ]:
for gas in gwp100.index :
    rescale = gwp100.loc[gas, 'AR4-5-rescale']
    crf_2022_cb0.loc[gas,:] *= rescale

crf_2022_cb0['CB0'] = crf_2022_cb0.drop('CB0',axis='columns').sum(axis='columns')

crf_2022_cb0.loc['Total-AR5'] = crf_2022_cb0.drop(['Total ','Total-xchk'],axis='index').sum(axis='index')

In [ ]:
display(crf_2022_cb0)

In [ ]:
infile = "EPA-Inv-Final-1990-2020/GHG_Final-emissions-data_1990-2020_AR4_web.xlsx"
epa_final_1990_2020 = pd.DataFrame()
epa_final_1990_2020 = pd.read_excel(infile, sheet_name="NEW Summary 1990-2020 GHG", header=0, index_col=0)

In [ ]:
display_all(epa_final_1990_2020)

In [ ]:
epa_final_1990_2020_cb0 = epa_final_1990_2020.loc['National Total with LULUCF',2016:2020]

In [ ]:
display(epa_final_1990_2021_cb0)